[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C05/E05-2_Resistance_Composite.ipynb)

# Résistance d'un matériau composite : retenir l'essentiel par la régularisation

## Le contexte

Un laboratoire fabrique des plaques de matériau composite en faisant varier huit réglages de procédé : le ratio fibres sur matrice, la température de cuisson, la pression de compactage, la durée de polymérisation, le taux d'humidité de la résine, la densité des fibres, la vitesse de refroidissement et le taux d'additifs. Chaque plaque est ensuite testée en traction. On voudrait prévoir la résistance à la traction d'après les réglages, pour ne plus fabriquer que les plaques prometteuses.

Plusieurs réglages évoluent ensemble, et un modèle linéaire a du mal à répartir leur effet. La régularisation sert à garder l'essentiel : elle freine les coefficients, et le lasso peut même en annuler.

La grandeur à prédire est un nombre en MPa : c'est un problème de régression.

Défi : prédire `resistance_traction_MPa` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression ridge sur les huit réglages mis à l'échelle ; il atteint un R² de 0,646 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Resistance_Composite.xlsx`, dans le dossier `donnees` du dépôt : 2 000 plaques, 9 colonnes. Une ligne est une plaque.

| Colonne | Signification | Unité |
|---|---|---|
| `ratio_fibres_matrice` | ratio de fibres sur matrice | sans unité déclarée |
| `temperature_cuisson` | température de cuisson | sans unité déclarée |
| `pression_compactage` | pression de compactage | sans unité déclarée |
| `duree_polymerisation` | durée de polymérisation | sans unité déclarée |
| `taux_humidite_resine` | taux d'humidité de la résine | sans unité déclarée |
| `densite_fibres` | densité des fibres | sans unité déclarée |
| `vitesse_refroidissement` | vitesse de refroidissement | sans unité déclarée |
| `taux_additifs` | taux d'additifs | sans unité déclarée |
| `resistance_traction_MPa` | cible : résistance à la traction, écrêtée à 700 | MPa |

Origine : jeu public transformé. Un échantillon de 2 000 lignes dont les huit variables sont réinterprétées en réglages de procédé, la grandeur à prédire devenant la résistance, écrêtée à 700 MPa. Il enseigne une méthode, et aucune conclusion de matériau n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, les échelles des réglages et l'écrêtage de la cible avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Resistance_Composite.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Resistance_Composite.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "resistance_traction_MPa"
reglages = [colonne for colonne in df.columns if colonne != cible]
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
print(df[reglages].agg(["min", "max", "std"]).T.round(2).to_string())
ecretees = int((df[cible] >= 700).sum())
print(f"résistances écrêtées à 700 MPa : {ecretees}")

Lecture du résultat : 2 000 plaques, 9 colonnes, aucune case vide. Les huit réglages n'ont pas la même échelle : l'écart-type va de 0,6 à 125. La régularisation pénalise la taille des coefficients, qui dépend de l'échelle de chaque colonne : il faudra mettre les réglages à l'échelle. La résistance est écrêtée à 700 MPa pour 106 plaques, soit 5,3 % : le modèle ne verra jamais de valeur plus haute.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Resistance_Composite.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des plaques est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "plaques")
print("test :", len(X_test), "plaques")

### Étape 3 : chercher les réglages qui évoluent ensemble

Deux réglages très corrélés portent presque la même information. Le tableau donne, sur l'apprentissage seul, les trois paires les plus corrélées.

In [ ]:
correlations = X_train.corr(numeric_only=True)
paires = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack()
plus_liees = paires.loc[paires.abs().sort_values(ascending=False).head(3).index]
print(plus_liees.round(2).to_string())

Lecture du résultat : la paire la plus liée est `vitesse_refroidissement` et `taux_additifs`, avec une corrélation de -0,93 : quand l'un monte, l'autre baisse. Un modèle linéaire a du mal à départager de tels réglages : un coefficient peut compenser l'autre sans améliorer la prédiction.

### Étape 4 : construire le modèle de départ

Le modèle de départ est celui de la section du livre : une régression ridge, qui ajoute à la somme des carrés des écarts une pénalité proportionnelle à la somme des carrés des coefficients, avec son réglage par défaut (`alpha=1`). Les réglages sont mis à l'échelle d'abord, sur l'apprentissage seul. Le pipeline apprend sur `X_train`.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les plaques qu'il n'a jamais vues. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en MPa, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin sans apprentissage, qui répond toujours la moyenne de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.1f} MPa")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.1f} MPa")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")
coefficients = pd.Series(modele[-1].coef_, index=X_train.columns)
print("coefficients (MPa par écart-type du réglage) :")
print(coefficients.sort_values(key=abs, ascending=False).round(1).to_string())

Lecture du résultat : le R² est de 0,646 et l'erreur absolue moyenne de 54,0 MPa, pour une résistance moyenne de 414 MPa sur la base de test. Le témoin, qui répond toujours la moyenne, n'explique rien : son R² est nul, au signe de l'arrondi près. Les coefficients confirment la lecture de l'étape 3 : `vitesse_refroidissement` et `taux_additifs` reçoivent des coefficients de même signe et de grande taille, -83,4 et -77,2 MPa par écart-type, alors que ces deux réglages varient en sens contraire : leurs effets se compensent en grande partie. C'est ce chiffre, mesuré sur des plaques que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : choisir la régularisation, puis les termes à garder

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première compare par validation croisée, sur l'apprentissage seul, le ridge, le lasso et l'elastic net sur les huit réglages, puis les mêmes méthodes sur des polynômes de degré 2 et 3, dont la régression sans pénalité. Le coefficient de pénalité est choisi à l'intérieur de chaque pli. La seconde mesure le modèle retenu une seule fois sur le test et compte les termes qu'il garde. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Des réglages très corrélés reçoivent des coefficients de grande taille qui se compensent : le modèle est instable, sans que le score le dise.
2. Le ridge freine tous les coefficients sans en annuler, le lasso en annule et sélectionne, l'elastic net mélange les deux ; la pénalité s'applique à des colonnes mises à l'échelle.
3. Sur les huit réglages bruts, la régularisation ne change presque pas le score (0,680 sans pénalité, 0,680 avec le ridge, en validation croisée) : elle stabilise le modèle, elle ne lui ajoute pas de pouvoir de prédiction.
4. Elle devient indispensable quand on ajoute des termes : un polynôme de degré 3 compte 164 colonnes pour 1 500 plaques, et sans pénalité l'écart entre les plis passe à 0,11, contre 0,04 avec le lasso.
5. La force de la pénalité se choisit par validation croisée sur l'apprentissage ; le test ne sert qu'une fois, au modèle retenu.